# Attention from scratch

We're going to build a single self-attention block in NumPy and then extend it to multi-head. The point is to see that the whole mechanism is about a dozen lines of linear algebra.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(47)

## 1. The problem

Consider a clinical note: *"The patient discontinued metformin because it caused nausea."*

What does "it" refer to? You can easily see that this means the drug, not the patient. But for a model extracting adverse drug reactions from clinical text, getting this wrong means characterizing the side effect incorrectly.

We previously learned about RNNs and LSTMS, but both have to carry "metformin" forward through through the network until it gets to "it", and that signal degrades over distance. In a real note the relevant information might be sentences or even pages away.

Attention's approach: when processing "it", look at every token in the sequence at once and weight each one by relevance.

## 2. Tokens and embeddings

Before we encode the attention mechanism, we need to make an embedding of each token. As we talked about in lecture, in a real model these are learned; here we'll use random vectors.

Six tokens, embedding dimension 8. We're stopping the sentence at "it" for simplicity.

In [ ]:
tokens = ["The", "patient", "discontinued", "metformin", "because", "it"]
n_tokens = len(tokens)
d_model = 8

X = np.random.randn(n_tokens, d_model)

print(f"X shape: {X.shape}")

for i, token in enumerate(tokens):
    print(token, "->", X[i].round(2))

# Could also add positional embedding.
# This is often an element-wise summation: Final input = token embedding + positional embedding
# In this case token embedding size = positional embeding size (in this case 8)

## 3. Calculating attention

In lecture, we left off with the following equation: 

$$Y = \text{softmax}\!\left(\frac{QK^T}{\sqrt{E}}\right) V$$

Visual of the equation:<br><br>
<img src="image.png" width="200">

### What are Q, K, and V?

For each token we compute three vectors via three different linear projections:

- **Q (query):** what this token is looking for
- **K (key):** what this token offers as a match target
- **V (value):** the information this token carries

**E:** embedding dimension
**Y:** Attention output

### How are they calculated?
$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

$W_Q, W_K, W_V$ are learnable weight matrices of shape `(d_model, d_model)`.

In [ ]:
# These are learned weights. For now, we will initialize them randomly
W_Q = np.random.randn(d_model, d_model) / np.sqrt(d_model)
W_K = np.random.randn(d_model, d_model) / np.sqrt(d_model)
W_V = np.random.randn(d_model, d_model) / np.sqrt(d_model)

Plot Q, K, V side by side. Each row is a token, each column is a dimension. Same input, three different views.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

for ax, mat, name in zip(axes, [Q, K, V], ["Q (queries)", "K (keys)", "V (values)"]):
    sns.heatmap(mat, ax=ax, cmap="RdBu_r", center=0,
                yticklabels=tokens, xticklabels=range(d_model))
    ax.set_title(name)
    ax.set_xlabel("dimension")

plt.tight_layout()

## 4. Attention scores

How much should token $i$ attend to token $j$? Take the dot product of $i$'s query with $j$'s key:

$$\text{score}(i, j) = Q_i \cdot K_j$$

Dot products measure alignment: vectors pointing the same direction give large positive values, orthogonal vectors give zero. So tokens whose query matches another token's key get high scores.

Let's take a look at one example:

In [ ]:
# The Q vector of 'The' alignment with the K vector of 'patient'


Now let's do all values together:

### Scaling by ${\sqrt{E}}$

The full formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{E}}\right) V$$

Dividing by $\sqrt{E}$ keeps the variance approximately constant regardless of embedding size.

## 5. Softmax

Convert scores to non-negative weights that sum to 1 along each row, so we can read off "what fraction of attention does token $i$ pay to each other token":

Apply row-by-row, since each query distributes its attention independently.

In [ ]:
def softmax(x, axis=-1):
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


Plot the attention matrix. Cell `(i, j)` is how much query token $i$ attends to key token $j$. Read row by row.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(attention_weights, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=tokens, yticklabels=tokens, cbar_kws={"shrink": 0.8},
            ax=ax)
ax.set_xlabel("Key (attending TO)")
ax.set_ylabel("Query (attending FROM)")
ax.set_title("Attention weights")
plt.tight_layout()
plt.show()

## 6. Weighted sum of values

Now we use the attention weights to take a weighted average of the value vectors. The new representation of token $i$ is:

$$\text{output}_i = \sum_j \text{attention}_{ij} \cdot V_j$$

Let's start with the first word, *"The"*.

First, look at the attention weights for *"The"* (row 0 of the attention matrix). These tell us how much *"The"* attends to each of the six tokens:

In [ ]:
for token, w in zip(tokens, attention_weights[0]):
    print(f"  attention from 'The' to '{token}': {w:.3f}")

print(f"\nrow sum: {attention_weights[0].sum():.3f}")

Now let's compute *one* dimension of the new representation of *"The"*. Take dimension 0: the new value is the weighted sum of dimension 0 of every token's value vector.

$$\text{output}_{\text{The}, \, 0} = \sum_j \text{attention}_{\text{The}, j} \cdot V_{j, 0}$$

We do the same weighted sum for every dimension. Doing all dimensions at once is just `attention_weights[0] @ V`:

We've turned *"The"* into a vector that has absorbed information from every other token in the sentence, weighted by relevance.

Doing this for every token at once is one matrix multiply:

$$\text{output} = \text{attention\_weights} \cdot V$$

In [ ]:
for i, token in enumerate(tokens):
    print(token, "->", output[i].round(2))

## 7. Attention as one function

Putting steps 3-6 together:

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q: (..., n_q, d_k)
        K: (..., n_k, d_k)
        V: (..., n_k, d_v)
        mask: optional bool array (..., n_q, n_k); True = blocked
    Returns:
        output: (..., n_q, d_v)
        attention_weights: (..., n_q, n_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    
    if mask is not None:
        scores = np.where(mask, -np.inf, scores)
    
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

out, w = scaled_dot_product_attention(Q, K, V)
print("matches step-by-step:", np.allclose(out, output))

Every transformer is built around variations of this. The rest of the architecture (layer norm, residuals, FFN, positional encodings) is scaffolding around this.

## 8. Multi-head attention

A single attention computation can only learn one kind of token-to-token relationship. In practice we want the model to capture several at once, similar to how multiple convolution filters capture different patterns in an image in a CNN. 

Multi-head attention runs $h$ attention computations in parallel, each on a slice of the embedding dimensions, then concatenates the results. Same total compute as one big attention, more expressive.

In [ ]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O, n_heads):
    """
    Args:
        X: (n_tokens, d_model)
        W_Q, W_K, W_V: (d_model, d_model)
        W_O: (d_model, d_model)  output projection
        n_heads: int, must divide d_model
    """
    n_tokens, d_model = X.shape
    d_k = d_model // n_heads
    
    # 1. project
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    
    # 2. split into heads: (n_tokens, n_heads, d_k) -> (n_heads, n_tokens, d_k)
    Q = Q.reshape(n_tokens, n_heads, d_k).transpose(1, 0, 2)
    K = K.reshape(n_tokens, n_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(n_tokens, n_heads, d_k).transpose(1, 0, 2)
    
    # 3. attention per head, vectorized
    head_outputs, all_weights = scaled_dot_product_attention(Q, K, V)
    
    # 4. concatenate heads back: (n_tokens, d_model)
    concat = head_outputs.transpose(1, 0, 2).reshape(n_tokens, d_model)
    
    # 5. output projection
    output = concat @ W_O
    return output, all_weights

n_heads = 4
W_Q_full = np.random.randn(d_model, d_model) / np.sqrt(d_model)
W_K_full = np.random.randn(d_model, d_model) / np.sqrt(d_model)
W_V_full = np.random.randn(d_model, d_model) / np.sqrt(d_model)
W_O_full = np.random.randn(d_model, d_model) / np.sqrt(d_model)

mha_output, head_weights = multi_head_attention(X, W_Q_full, W_K_full, W_V_full, W_O_full, n_heads)

print(f"output shape:        {mha_output.shape}")
print(f"per-head weights:    {head_weights.shape}")

Each head learns its own attention pattern.

In [ ]:
fig, axes = plt.subplots(1, n_heads, figsize=(13, 3))

for h in range(n_heads):
    sns.heatmap(head_weights[h], annot=True, fmt=".2f", cmap="viridis",
                xticklabels=tokens, yticklabels=tokens, cbar=False,
                ax=axes[h], vmin=0, vmax=1)
    axes[h].set_title(f"Head {h}")
    if h == 0:
        axes[h].set_ylabel("Query")
    axes[h].set_xlabel("Key")

plt.tight_layout()
plt.show()

## 9. Causal masking

For many generative LLMs, token $t$ must not attend to tokens $t+1, t+2, ...$. During training that would be cheating; at inference time those tokens don't exist yet.

Implementation: set the upper-triangular entries of the score matrix to $-\infty$ before softmax. After softmax those positions are exactly 0.

In [ ]:
causal_mask = np.triu(np.ones((n_tokens, n_tokens), dtype=bool), k=1)

print("mask (True = blocked):")
print(causal_mask.astype(int))

causal_out, causal_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.heatmap(attention_weights, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=tokens, yticklabels=tokens, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("No mask (bidirectional)")
axes[0].set_xlabel("Key");  axes[0].set_ylabel("Query")

sns.heatmap(causal_weights, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=tokens, yticklabels=tokens, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Causal mask (autoregressive)")
axes[1].set_xlabel("Key");  axes[1].set_ylabel("Query")

plt.tight_layout()
plt.show()

## 10. Same code, on a SMILES string

Attention doesn't care what the tokens are — text, amino acids, SMILES characters, image patches, all of it works. As a sanity check, run the same `multi_head_attention` on aspirin: `CC(=O)Oc1ccccc1C(=O)O`.

A real chemistry model uses a learned tokenizer (often byte-pair encoding over SMILES, or a chemistry-aware tokenizer that respects atoms and bonds). Character-level is fine here for showing that the mechanism doesn't change.

In [ ]:
smiles = "CC(=O)Oc1ccccc1C(=O)O"
smiles_tokens = list(smiles)
n_smiles = len(smiles_tokens)

print(f"SMILES: {smiles}")
print(f"{n_smiles} tokens")

X_smiles = np.random.randn(n_smiles, d_model)

smiles_out, smiles_weights = multi_head_attention(
    X_smiles, W_Q_full, W_K_full, W_V_full, W_O_full, n_heads
)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(smiles_weights[0], cmap="viridis",
            xticklabels=smiles_tokens, yticklabels=smiles_tokens,
            cbar_kws={"shrink": 0.7}, ax=ax, vmin=0, vmax=smiles_weights[0].max())
ax.set_title("Aspirin SMILES, head 0 (untrained)")
ax.set_xlabel("Key")
ax.set_ylabel("Query")
plt.tight_layout()
plt.show()

Same Q/K/V, dot product, softmax, weighted sum. With trained weights you'd see related atoms with high attention scores: ring atoms attending to other ring atoms, functional groups clustering etc.